# Step 16 — Cancer-specific unsupervised subclustering of `Tumor/epithelial` + `Other/unresolved` (selection-fix v2)

This notebook performs a deeper, unsupervised analysis of the broad cancer
compartment in the 12-sample Visium HD dataset.

## Candidate universe

Only cells currently labeled:

```text
Tumor/epithelial
Other/unresolved
```

in:

```python
obs["prelim_cell_type_primary_tumor_expanded"]
```

are included.

Resolved immune, myeloid, fibroblast/stromal, endothelial, and T-cell labels
are not reclustered and are never overwritten.

## Why cluster separately by cancer type?

Melanoma, NSCLC, and MSS-CRC have very different lineage programs. Clustering
them together would mostly recover tissue-of-origin differences. The default
analysis therefore creates three independent objects:

```text
melanoma      3 paired patients / 6 samples
NSCLC        2 paired patients / 4 samples
colon_cancer 1 paired patient  / 2 samples
```

## Expression and count conventions

The source is the completed Step 15 merged AnnData Zarr:

```python
source.X
# dense float32 ResolVI-corrected expression
# normalized to library size 10,000
# not log-transformed

source.layers["counts"]
# sparse observed raw integer Proseg counts
```

For clustering, this notebook stores:

```python
candidate.X = log1p(ResolVI-corrected 10k expression)
candidate.layers["counts"] = observed raw counts
```

Raw counts remain available for pseudobulk, count-based DE, and raw-detection
audits.

## Unsupervised design

For each cancer type:

1. lazily subset the merged Zarr to the two candidate labels;
2. optionally require `qc_pass`;
3. select sample-aware HVGs from corrected expression;
4. require minimal raw-count support for PCA features;
5. exclude mitochondrial, ribosomal, hemoglobin, deprecated, and cell-cycle
   genes from the identity PCA;
6. run PCA;
7. save an unintegrated UMAP for audit;
8. run sample-level Harmony by default;
9. build the neighbor graph and UMAP;
10. run Leiden at every requested resolution:

```text
0.1, 0.2, 0.3, 0.4, 0.5,
0.6, 0.7, 0.8, 0.9, 1.0
```

## Keratinocytes

This notebook does **not** run the separate signature-based keratinocyte
refinement and does not assign a `Keratinocyte` label. The previously observed
`DSP`, `DMKN`, `DSG1`, `DSC3` program is included only in the marker-audit
panel so that a naturally emerging cluster can be recognized.

## Main outputs per cancer type

```text
<cancer>_tumor_unresolved_multires_leiden.zarr
<cancer>_tumor_unresolved_cluster_assignments.parquet
<cancer>_resolution_summary.csv
<cancer>_cluster_summary_all_resolutions.csv
<cancer>_adjacent_resolution_ARI.csv
<cancer>_cluster_sample_composition.csv
<cancer>_cluster_source_label_composition.csv
<cancer>_primary_resolution_markers.csv
<cancer>_umap_all_resolutions.png
<cancer>_umap_audit_views.png
<cancer>_<sample>_spatial_primary_resolution.png
```

The cluster-assignment Parquet is the lightweight file to join back to the
full all-cell object.


## Source-branch provenance

The merged input was exported from:

```text
12_broader_tumor_epithelial_rescue/
```

and not from the separate `12B_keratinocyte_refinement` branch.


## Selection-fix v2

This revision does not rely on exact raw string equality for cancer type, broad
label, or QC-pass values. Cancer type is derived primarily from the explicit
`SAMPLE_INFO` mapping, while the source `obs["cancer_type"]` is retained and
audited. Broad labels are normalized only for the two requested candidate
classes, and `qc_pass` accepts Boolean, numeric, and common string forms. A
gate-by-gate inventory is written before any dense expression is loaded.


In [1]:
# ---------------------------------------------------------------------
# GPU selection — run before importing CuPy/RAPIDS
# ---------------------------------------------------------------------
import os

GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("OMP_NUM_THREADS", "16")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "16")
os.environ.setdefault("MKL_NUM_THREADS", "16")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "16")

print("CUDA_VISIBLE_DEVICES:", GPU_ID)


CUDA_VISIBLE_DEVICES: 1


In [2]:
# ---------------------------------------------------------------------
# Imports and environment checks
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import inspect
import importlib.metadata as importlib_metadata
import json
import math
import re
import shutil
import time
import traceback
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import zarr

from IPython.display import display
from sklearn.metrics import adjusted_rand_score

try:
    import cupy as cp
    import rapids_singlecell as rsc

    RAPIDS_AVAILABLE = True
    RAPIDS_IMPORT_ERROR = ""
except Exception as exc:
    cp = None
    rsc = None
    RAPIDS_AVAILABLE = False
    RAPIDS_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

warnings.filterwarnings("ignore", category=FutureWarning)
plt.ioff()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


print("Python:", os.sys.version)
print("anndata:", package_version("anndata"))
print("scanpy:", package_version("scanpy"))
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", package_version("scipy"))
print("zarr:", package_version("zarr"))
print("rapids-singlecell:", package_version("rapids-singlecell"))
print("RAPIDS available:", RAPIDS_AVAILABLE)

if RAPIDS_AVAILABLE:
    print("Visible CUDA devices:", cp.cuda.runtime.getDeviceCount())
    print(
        "GPU:",
        cp.cuda.runtime.getDeviceProperties(0)["name"].decode(),
    )
else:
    print("RAPIDS import error:", RAPIDS_IMPORT_ERROR)


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.13.7 (main, Sep 18 2025, 19:47:49) [Clang 20.1.4 ]
anndata: 0.12.11
scanpy: 1.12.1
numpy: 2.4.4
pandas: 2.3.3
scipy: 1.17.1
zarr: 3.1.6
rapids-singlecell: not installed
RAPIDS available: True
Visible CUDA devices: 1
GPU: NVIDIA A10G


In [3]:
# ---------------------------------------------------------------------
# Study metadata and user configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

# This is the Step 12 merged object, not the Step 12B keratinocyte-refined
# branch. If this exact path is absent, the preflight cell searches for the
# filename recursively under PIPELINE_ROOT.
MERGED_ZARR = (
    PIPELINE_ROOT
    / "15_merged_annotation_handoff"
    / "VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr"
)

OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "16_tumor_unresolved_leiden_multiresolution_selectionfix_v2"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen",
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15",
    },
}

CANCER_TYPES_TO_RUN = [
    "melanoma",
    "NSCLC",
    "colon_cancer",
]

# To use three GPUs in parallel, open three copies/kernels:
# GPU 0: GPU_ID="0"; CANCER_TYPES_TO_RUN=["melanoma"]
# GPU 1: GPU_ID="1"; 
CANCER_TYPES_TO_RUN=["NSCLC"]
# GPU 2: GPU_ID="2"; CANCER_TYPES_TO_RUN=["colon_cancer"]

ANNOTATION_COLUMN = (
    "prelim_cell_type_primary_tumor_expanded"
)
CANDIDATE_LABELS = {
    "Tumor/epithelial",
    "Other/unresolved",
}

# Internal, audit-friendly selection columns created during preflight.
SELECTION_CANCER_COLUMN = "_cancer_type_for_clustering"
SELECTION_LABEL_COLUMN = "_candidate_label_for_clustering"
SELECTION_QC_COLUMN = "_qc_pass_for_clustering"
SOURCE_CANCER_COLUMN = "source_cancer_type_before_selection_normalization"
SOURCE_LABEL_COLUMN = "source_annotation_before_selection_normalization"

# Only aliases of the two requested broad classes are canonicalized. Other
# labels are retained exactly as observed and remain outside the candidate set.
CANDIDATE_LABEL_TOKEN_MAP = {
    "tumorepithelial": "Tumor/epithelial",
    "tumorepithelium": "Tumor/epithelial",
    "tumororepithelial": "Tumor/epithelial",
    "otherunresolved": "Other/unresolved",
    "unresolvedother": "Other/unresolved",
}

TRUE_QC_TOKENS = {
    "true", "t", "1", "1.0", "yes", "y", "pass", "passed",
}
FALSE_QC_TOKENS = {
    "false", "f", "0", "0.0", "no", "n", "fail", "failed", "",
    "nan", "none", "missing",
}

# A Step 12B-refined merged object would contain an explicit Keratinocyte
# label, causing those cells to be omitted from this candidate universe.
# Keep this False so the notebook stops rather than silently using that branch.
ALLOW_KERATINOCYTE_REFINED_INPUT = False

REQUIRE_QC_PASS = True
QC_PASS_COLUMN = "qc_pass"

# Exact requested resolution series.
LEIDEN_RESOLUTIONS = [
    round(value / 10, 1)
    for value in range(1, 11)
]
PRIMARY_RESOLUTION = 0.6

# Corrected-expression feature space.
N_TOP_HVG = 3_000
HVG_MAX_CELLS = 100_000
HVG_MAX_CELLS_PER_SAMPLE = 20_000
PCA_N_COMPS = 50
PCA_SCALE_MAX_VALUE = 10.0
RANDOM_STATE = 0

# Require a feature to be observed in raw counts in at least this many
# candidate cells, or the configured fraction, whichever is larger.
MIN_RAW_CELLS_PER_GENE = 20
MIN_RAW_CELL_FRACTION_PER_GENE = 0.0005

EXCLUDE_CELL_CYCLE_FROM_IDENTITY_PCA = True

# The default matches the established cancer-specific integration workflow.
# Set APPLY_HARMONY=False to preserve all sample-specific variation.
APPLY_HARMONY = True
HARMONY_BATCH_KEY = "sample"
HARMONY_THETA = 2.0
HARMONY_TAU = 20
HARMONY_MAX_ITER = 20
FAIL_IF_HARMONY_FAILS = True

SAVE_UNINTEGRATED_UMAP = True

N_NEIGHBORS = 15
N_PCS_NEIGHBORS = 30
NEIGHBOR_ALGORITHM = "brute"
UMAP_MIN_DIST = 0.30
UMAP_SPREAD = 1.0
LEIDEN_N_ITERATIONS = 100

REQUIRE_RAPIDS = True
ALLOW_CPU_FALLBACK = False

# Marker ranking is performed only at the primary resolution by default.
RUN_PRIMARY_MARKER_RANKING = True
N_MARKER_GENES = 100
MAX_MARKER_CELLS_PER_CLUSTER = 5_000

WRITE_FULL_ZARR = True
WRITE_H5AD = False
H5AD_COMPRESSION = "lzf"
ZARR_CHUNKS = (512, 2_048)

OVERWRITE_OUTPUTS = False
USE_EXISTING_COMPLETED = True
CONTINUE_ON_ERROR = True

PLOT_DPI = 300
PLOT_MAX_CELLS = 200_000

PIPELINE_VERSION = (
    "2026-08-20-tumor-unresolved-cancer-specific-leiden-v2-selectionfix"
)

print("Cancer types:", CANCER_TYPES_TO_RUN)
print("Leiden resolutions:", LEIDEN_RESOLUTIONS)
print("Output root:", OUTPUT_ROOT)


Cancer types: ['NSCLC']
Leiden resolutions: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2


## Preflight

The source object is deliberately the Step 15 merge of the original Step 12
annotation. It is **not** the Step 12B keratinocyte-refined object.

The preflight confirms:

```text
.X                       corrected 10k expression
.layers["counts"]         raw Proseg counts
.obsm["spatial"]          sample-local coordinates
.obs[annotation column]   broad preliminary labels
```

It also writes a candidate inventory before loading the dense expression
matrix.


In [4]:
# ---------------------------------------------------------------------
# Input discovery, lazy validation, and selection-gate inventory
# ---------------------------------------------------------------------
EXPECTED_MERGED_NAME = (
    "VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr"
)

if not MERGED_ZARR.exists():
    candidates = sorted(
        PIPELINE_ROOT.rglob(EXPECTED_MERGED_NAME)
    )
    if len(candidates) == 1:
        MERGED_ZARR = candidates[0]
        print("Auto-discovered merged Zarr:", MERGED_ZARR)
    elif len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find {EXPECTED_MERGED_NAME} under {PIPELINE_ROOT}"
        )
    else:
        raise RuntimeError(
            "Multiple merged Zarr candidates were found. Set MERGED_ZARR "
            "explicitly:\n"
            + "\n".join(str(path) for path in candidates)
        )

if not hasattr(ad.experimental, "read_lazy"):
    raise RuntimeError(
        "This notebook requires anndata.experimental.read_lazy."
    )

merged_lazy = ad.experimental.read_lazy(
    str(MERGED_ZARR)
)


def to_memory_dataframe(frame) -> pd.DataFrame:
    if hasattr(frame, "to_memory"):
        frame = frame.to_memory()
    if hasattr(frame, "compute"):
        frame = frame.compute()
    return pd.DataFrame(frame).copy()


def clean_text_series(series: pd.Series) -> pd.Series:
    """Decode/strip common backed-string representations without changing case."""
    values = series.copy()

    def clean_one(value):
        if pd.isna(value):
            return ""
        if isinstance(value, bytes):
            value = value.decode("utf-8", errors="replace")
        text = str(value).strip()
        # Handle literal byte-string text such as "b'melanoma'".
        if (
            len(text) >= 3
            and text[0] == "b"
            and text[1] in {"'", '"'}
            and text[-1] == text[1]
        ):
            text = text[2:-1]
        return text.strip()

    return values.map(clean_one)


def normalized_token(value) -> str:
    """Case/spacing/punctuation-insensitive token for controlled aliases."""
    if pd.isna(value):
        return ""
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


def canonical_sample_series(series: pd.Series) -> pd.Series:
    cleaned = clean_text_series(series)

    def canonical(value):
        if value in SAMPLE_INFO:
            return value
        if value.startswith("Sample_") and value[7:] in SAMPLE_INFO:
            return value[7:]
        return value

    return cleaned.map(canonical)


def canonical_candidate_label_series(series: pd.Series) -> pd.Series:
    cleaned = clean_text_series(series)

    def canonical(value):
        return CANDIDATE_LABEL_TOKEN_MAP.get(
            normalized_token(value),
            value,
        )

    return cleaned.map(canonical)


def coerce_qc_pass(series: pd.Series) -> pd.Series:
    """Robustly parse Boolean, numeric, and common text QC-pass values."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        return numeric.fillna(0).ne(0)

    cleaned = clean_text_series(series)
    tokens = cleaned.map(normalized_token)
    parsed = pd.Series(False, index=series.index, dtype=bool)
    parsed.loc[tokens.isin(TRUE_QC_TOKENS)] = True

    unknown = sorted(
        set(tokens.unique())
        - TRUE_QC_TOKENS
        - FALSE_QC_TOKENS
    )
    if unknown:
        warnings.warn(
            "Unrecognized qc_pass token(s) were treated as False: "
            f"{unknown[:20]}"
        )
    return parsed


merged_obs = to_memory_dataframe(
    merged_lazy.obs
)
merged_obs.index = merged_obs.index.astype(str)

required_obs = {
    "sample",
    "patient",
    "cancer_type",
    "biopsy_stage",
    ANNOTATION_COLUMN,
}
missing_obs = sorted(
    required_obs.difference(merged_obs.columns)
)
if missing_obs:
    raise KeyError(
        f"Merged obs is missing required columns: {missing_obs}"
    )

if "counts" not in merged_lazy.layers:
    raise KeyError(
        "Merged object lacks layers['counts']."
    )
if "spatial" not in merged_lazy.obsm:
    raise KeyError(
        "Merged object lacks obsm['spatial']."
    )
if merged_obs.index.duplicated().any():
    raise ValueError(
        "Merged obs_names are not unique."
    )

# Preserve exact source strings for audit.
merged_obs["sample"] = canonical_sample_series(
    merged_obs["sample"]
)
merged_obs[SOURCE_CANCER_COLUMN] = clean_text_series(
    merged_obs["cancer_type"]
)
merged_obs[SOURCE_LABEL_COLUMN] = clean_text_series(
    merged_obs[ANNOTATION_COLUMN]
)
merged_obs["patient"] = clean_text_series(
    merged_obs["patient"]
)
merged_obs["biopsy_stage"] = clean_text_series(
    merged_obs["biopsy_stage"]
)

# Derive cancer type from the explicit sample map. This is more reliable than
# exact matching of serialized category strings, while the source value remains
# available for discrepancy audits.
sample_to_cancer = {
    sample: metadata["cancer_type"]
    for sample, metadata in SAMPLE_INFO.items()
}
merged_obs[SELECTION_CANCER_COLUMN] = (
    merged_obs["sample"].map(sample_to_cancer)
)

unmapped_sample = merged_obs[SELECTION_CANCER_COLUMN].isna()
if unmapped_sample.any():
    examples = sorted(
        merged_obs.loc[unmapped_sample, "sample"].unique()
    )[:20]
    raise RuntimeError(
        "Some cells could not be mapped to a study cancer type from sample: "
        f"{examples}"
    )

merged_obs[SELECTION_LABEL_COLUMN] = (
    canonical_candidate_label_series(
        merged_obs[SOURCE_LABEL_COLUMN]
    )
)

if REQUIRE_QC_PASS:
    if QC_PASS_COLUMN not in merged_obs:
        raise KeyError(
            f"REQUIRE_QC_PASS=True but {QC_PASS_COLUMN!r} is absent."
        )
    merged_obs[SELECTION_QC_COLUMN] = coerce_qc_pass(
        merged_obs[QC_PASS_COLUMN]
    )
else:
    merged_obs[SELECTION_QC_COLUMN] = True

unknown_samples = sorted(
    set(merged_obs["sample"]).difference(SAMPLE_INFO)
)
if unknown_samples:
    raise RuntimeError(
        f"Samples not present in SAMPLE_INFO: {unknown_samples}"
    )

observed_primary_labels = sorted(
    merged_obs[SOURCE_LABEL_COLUMN]
    .dropna()
    .astype(str)
    .unique()
)
print("Observed primary labels:", observed_primary_labels)

if (
    any(normalized_token(value) == "keratinocyte" for value in observed_primary_labels)
    and not ALLOW_KERATINOCYTE_REFINED_INPUT
):
    raise RuntimeError(
        "The merged object contains an explicit Keratinocyte label and "
        "therefore appears to come from the Step 12B refinement branch. "
        "Point MERGED_ZARR to the original Step 12 merged export, or "
        "deliberately set ALLOW_KERATINOCYTE_REFINED_INPUT=True."
    )

# Source-versus-sample-map cancer-type audit.
source_cancer_audit = (
    merged_obs.groupby(
        [
            "sample",
            SOURCE_CANCER_COLUMN,
            SELECTION_CANCER_COLUMN,
        ],
        dropna=False,
        observed=True,
    )
    .size()
    .rename("n_cells")
    .reset_index()
)
source_cancer_audit.to_csv(
    OUTPUT_ROOT / "source_vs_sample_mapped_cancer_type.csv",
    index=False,
)
display(source_cancer_audit)

# Gate-by-gate audit. This is the first place to look if selection is empty.
gate_rows = []
for cancer_type in CANCER_TYPES_TO_RUN:
    cancer_mask = merged_obs[SELECTION_CANCER_COLUMN].eq(
        cancer_type
    )
    label_mask = merged_obs[SELECTION_LABEL_COLUMN].isin(
        CANDIDATE_LABELS
    )
    qc_mask = merged_obs[SELECTION_QC_COLUMN].astype(bool)

    gate_rows.append(
        {
            "cancer_type": cancer_type,
            "n_all_cells": int(len(merged_obs)),
            "n_cells_in_cancer_from_sample_map": int(cancer_mask.sum()),
            "n_candidate_labels_all_cancers": int(label_mask.sum()),
            "n_candidate_before_qc_in_cancer": int(
                (cancer_mask & label_mask).sum()
            ),
            "n_qc_pass_in_cancer": int(
                (cancer_mask & qc_mask).sum()
            ),
            "n_final_eligible": int(
                (cancer_mask & label_mask & qc_mask).sum()
            ),
        }
    )

selection_gate_audit = pd.DataFrame(gate_rows)
selection_gate_audit.to_csv(
    OUTPUT_ROOT / "selection_gate_audit.csv",
    index=False,
)
display(selection_gate_audit)

# Raw value counts make future serialization mismatches immediately visible.
for column, filename in [
    (SOURCE_CANCER_COLUMN, "source_cancer_type_value_counts.csv"),
    (SOURCE_LABEL_COLUMN, "source_annotation_value_counts.csv"),
    (QC_PASS_COLUMN, "source_qc_pass_value_counts.csv"),
]:
    if column in merged_obs:
        (
            merged_obs[column]
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="n_cells")
            .to_csv(OUTPUT_ROOT / filename, index=False)
        )

inventory_mask = (
    merged_obs[SELECTION_LABEL_COLUMN].isin(CANDIDATE_LABELS)
    & merged_obs[SELECTION_QC_COLUMN].astype(bool)
)

candidate_inventory = (
    merged_obs.loc[inventory_mask]
    .groupby(
        [
            SELECTION_CANCER_COLUMN,
            "sample",
            "patient",
            "biopsy_stage",
            SELECTION_LABEL_COLUMN,
        ],
        observed=True,
    )
    .size()
    .rename("n_cells")
    .reset_index()
    .rename(
        columns={
            SELECTION_CANCER_COLUMN: "cancer_type",
            SELECTION_LABEL_COLUMN: ANNOTATION_COLUMN,
        }
    )
)

candidate_inventory.to_csv(
    OUTPUT_ROOT / "candidate_inventory.csv",
    index=False,
)
display(candidate_inventory)

print("Merged shape:", merged_lazy.shape)
print("Merged Zarr:", MERGED_ZARR)
print("Total eligible cells:", int(inventory_mask.sum()))

zero_cancers = selection_gate_audit.loc[
    selection_gate_audit["n_final_eligible"] == 0,
    "cancer_type",
].tolist()
if zero_cancers:
    raise RuntimeError(
        "Selection preflight found zero eligible cells for: "
        f"{zero_cancers}. Review selection_gate_audit.csv and the three "
        "raw-value-count tables before loading corrected expression."
    )


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_core/xarray.py:32: UserWarning: Did not read zarr as consolidated. Consider consolidating your metadata.
  return func(*args, **kwargs)


Observed primary labels: ['B/plasma', 'CD4+ T', 'CD8+ T', 'Endothelial', 'Fibroblast/stromal', 'Monocyte/macrophage', 'NK', 'Other T', 'Other/unresolved', 'Treg', 'Tumor/epithelial']


,sample,source_cancer_type_before_selection_normalization,_cancer_type_for_clustering,n_cells
0,C2D15_16_22,melanoma,melanoma,76633
1,C2D15_17_26,NSCLC,NSCLC,87913
2,C2D15_18_23,melanoma,melanoma,18594
3,C2D15_23_25,colon_cancer,colon_cancer,10243
4,C2D15_30_16,melanoma,melanoma,351799
5,C2D15_39_21,NSCLC,NSCLC,10822
6,Screen_16_22,melanoma,melanoma,70438
7,Screen_17_26,NSCLC,NSCLC,47896
8,Screen_18_23,melanoma,melanoma,72386
9,Screen_23_25,colon_cancer,colon_cancer,25776


,cancer_type,n_all_cells,n_cells_in_cancer_from_sample_map,n_candidate_labels_all_cancers,n_candidate_before_qc_in_cancer,n_qc_pass_in_cancer,n_final_eligible
0,NSCLC,862427,169581,688528,140501,169581,140501


,cancer_type,sample,patient,biopsy_stage,prelim_cell_type_primary_tumor_expanded,n_cells
0,NSCLC,C2D15_17_26,patient_17_26,C2D15,Other/unresolved,63345
1,NSCLC,C2D15_17_26,patient_17_26,C2D15,Tumor/epithelial,12953
2,NSCLC,C2D15_39_21,patient_39_21,C2D15,Other/unresolved,7125
3,NSCLC,C2D15_39_21,patient_39_21,C2D15,Tumor/epithelial,1224
4,NSCLC,Screen_17_26,patient_17_26,Screen,Other/unresolved,32996
5,NSCLC,Screen_17_26,patient_17_26,Screen,Tumor/epithelial,7117
6,NSCLC,Screen_39_21,patient_39_21,Screen,Other/unresolved,12273
7,NSCLC,Screen_39_21,patient_39_21,Screen,Tumor/epithelial,3468
8,colon_cancer,C2D15_23_25,patient_23_25,C2D15,Other/unresolved,6090
9,colon_cancer,C2D15_23_25,patient_23_25,C2D15,Tumor/epithelial,1214


Merged shape: (862427, 14887)
Merged Zarr: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/15_merged_annotation_handoff/VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr
Total eligible cells: 688528


In [5]:
# ---------------------------------------------------------------------
# General helpers: paths, serialization, metadata, GPU cleanup
# ---------------------------------------------------------------------
def cancer_slug(cancer_type: str) -> str:
    return (
        str(cancer_type)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )


def resolution_key(resolution: float) -> str:
    text = f"{float(resolution):.1f}".replace(".", "_")
    return f"tumor_leiden_res_{text}"


def cancer_paths(cancer_type: str) -> dict[str, Path]:
    slug = cancer_slug(cancer_type)
    out = OUTPUT_ROOT / slug
    figures = out / "figures"
    tables = out / "tables"
    markers = out / "markers"
    spatial = figures / "spatial"

    for path in [out, figures, tables, markers, spatial]:
        path.mkdir(parents=True, exist_ok=True)

    prefix = f"{slug}_tumor_unresolved"
    return {
        "out": out,
        "figures": figures,
        "tables": tables,
        "markers": markers,
        "spatial": spatial,
        "zarr": out / f"{prefix}_multires_leiden.zarr",
        "h5ad": out / f"{prefix}_multires_leiden.h5ad",
        "assignments": (
            tables
            / f"{prefix}_cluster_assignments.parquet"
        ),
        "features": (
            tables
            / f"{prefix}_integration_features.csv"
        ),
        "resolution_summary": (
            tables
            / f"{prefix}_resolution_summary.csv"
        ),
        "cluster_summary": (
            tables
            / f"{prefix}_cluster_summary_all_resolutions.csv"
        ),
        "sample_composition": (
            tables
            / f"{prefix}_cluster_sample_composition.csv"
        ),
        "label_composition": (
            tables
            / f"{prefix}_cluster_source_label_composition.csv"
        ),
        "ari": (
            tables
            / f"{prefix}_adjacent_resolution_ARI.csv"
        ),
        "marker_table": (
            markers
            / f"{prefix}_primary_resolution_markers.csv"
        ),
        "raw_marker_detection": (
            markers
            / f"{prefix}_primary_resolution_raw_marker_detection.csv"
        ),
        "summary": (
            out / f"{prefix}_summary.json"
        ),
        "success": (
            out / f"{prefix}_SUCCESS.json"
        ),
        "failure": (
            out / f"{prefix}_FAILURE.json"
        ),
        "umap_resolutions": (
            figures
            / f"{prefix}_umap_all_resolutions.png"
        ),
        "umap_audit": (
            figures
            / f"{prefix}_umap_audit_views.png"
        ),
        "dotplot": (
            figures
            / f"{prefix}_primary_resolution_marker_dotplot.png"
        ),
    }


def write_json(payload, path: Path):
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        )
    )
    temporary.replace(path)


def to_builtin(value):
    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if value is None:
        return ""
    return value


def sanitize_anndata_for_write(
    adata_object: ad.AnnData,
) -> dict:
    converted = {}

    for frame_name, frame in [
        ("obs", adata_object.obs),
        ("var", adata_object.var),
    ]:
        converted_columns = []
        for column in frame.columns:
            series = frame[column]
            dtype_name = str(series.dtype)
            if (
                dtype_name.startswith("string")
                or dtype_name == "object"
            ):
                frame[column] = (
                    series.astype(object)
                )
                converted_columns.append(str(column))
        if converted_columns:
            converted[frame_name] = converted_columns

    adata_object.uns = to_builtin(
        dict(adata_object.uns)
    )
    return converted


def safe_write_h5ad(
    adata_object: ad.AnnData,
    path: Path,
):
    sanitize_anndata_for_write(adata_object)
    temporary = path.with_name(
        f"{path.stem}.tmp{path.suffix}"
    )
    temporary.unlink(missing_ok=True)

    adata_object.write_h5ad(
        temporary,
        compression=H5AD_COMPRESSION,
        convert_strings_to_categoricals=False,
    )
    temporary.replace(path)

    check = ad.read_h5ad(path, backed="r")
    try:
        if check.shape != adata_object.shape:
            raise RuntimeError(
                f"H5AD read-back shape changed: "
                f"{check.shape} vs {adata_object.shape}"
            )
    finally:
        check.file.close()


def free_gpu_memory():
    gc.collect()
    if RAPIDS_AVAILABLE:
        try:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass


def call_with_supported_kwargs(
    function,
    *args,
    **kwargs,
):
    signature = inspect.signature(function)
    parameters = signature.parameters
    accepts_var_kwargs = any(
        parameter.kind
        is inspect.Parameter.VAR_KEYWORD
        for parameter in parameters.values()
    )

    if accepts_var_kwargs:
        return function(*args, **kwargs)

    supported = {
        key: value
        for key, value in kwargs.items()
        if key in parameters
    }
    omitted = sorted(
        set(kwargs).difference(supported)
    )
    if omitted:
        print(
            f"{function.__name__}: omitted unsupported kwargs:",
            omitted,
        )
    return function(*args, **supported)


def numpy_from_gpu(value):
    if RAPIDS_AVAILABLE and isinstance(value, cp.ndarray):
        return cp.asnumpy(value)
    if hasattr(value, "get"):
        try:
            return value.get()
        except Exception:
            pass
    return np.asarray(value)


def categorical_string(series: pd.Series) -> pd.Categorical:
    return pd.Categorical(
        series.astype(str)
    )


print("General helpers ready.")


General helpers ready.


## Candidate loading and preprocessing

The merged Zarr is read lazily. Only one cancer type's candidate cells are
materialized at a time.

Feature selection is intentionally not driven by a curated tumor signature.
The tumor/keratinocyte marker panels are used only after clustering for
interpretation.

A gene must also have minimal raw-count support before it can enter the PCA.
This prevents a gene that exists only as diffuse corrected expression from
becoming an unsupervised identity feature.


In [6]:
# ---------------------------------------------------------------------
# Candidate loading, count validation, and technical-gene masks
# ---------------------------------------------------------------------
S_PHASE_GENES = {
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1",
    "UNG", "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1",
    "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1",
    "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3",
    "MSH2", "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6",
    "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2", "USP1",
    "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8",
}

G2M_PHASE_GENES = {
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2",
    "TOP2A", "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67",
    "TMPO", "CENPF", "TACC3", "FAM64A", "SMC4", "CCNB2",
    "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E",
    "TUBB4B", "GTSE1", "KIF20B", "HJURP", "CDCA3", "HN1",
    "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2",
    "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR",
    "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE",
    "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA",
}

HEMOGLOBIN_GENES = {
    "HBA1", "HBA2", "HBB", "HBD", "HBE1",
    "HBG1", "HBG2", "HBM", "HBQ1", "HBZ",
}


def load_candidate_cancer_type(
    cancer_type: str,
) -> ad.AnnData:
    cancer_mask = merged_obs[
        SELECTION_CANCER_COLUMN
    ].eq(cancer_type)
    label_mask = merged_obs[
        SELECTION_LABEL_COLUMN
    ].isin(CANDIDATE_LABELS)
    qc_mask = merged_obs[
        SELECTION_QC_COLUMN
    ].astype(bool)

    pre_qc_mask = cancer_mask & label_mask
    mask = pre_qc_mask & qc_mask

    n_cancer = int(cancer_mask.sum())
    n_pre_qc = int(pre_qc_mask.sum())
    n_qc_pass_cancer = int(
        (cancer_mask & qc_mask).sum()
    )
    n_selected = int(mask.sum())

    if n_selected < 50:
        source_label_counts = (
            merged_obs.loc[cancer_mask, SOURCE_LABEL_COLUMN]
            .astype(str)
            .value_counts()
            .head(20)
            .to_dict()
        )
        qc_value_counts = (
            merged_obs.loc[cancer_mask, QC_PASS_COLUMN]
            .astype(str)
            .value_counts(dropna=False)
            .head(20)
            .to_dict()
            if QC_PASS_COLUMN in merged_obs
            else {}
        )
        raise RuntimeError(
            f"{cancer_type}: only {n_selected} eligible cells after robust "
            "selection. Gate counts: cancer={n_cancer}, "
            f"candidate_before_qc={n_pre_qc}, "
            f"qc_pass_in_cancer={n_qc_pass_cancer}. "
            f"Top source labels={source_label_counts}. "
            f"qc_pass values={qc_value_counts}."
        )

    print(
        f"{cancer_type}: materializing {n_selected:,} candidate cells "
        f"({n_pre_qc:,} before QC)..."
    )

    selected_obs = merged_obs.loc[mask].copy()

    candidate = merged_lazy[
        mask.to_numpy(),
        :,
    ].to_memory()

    candidate.obs_names = candidate.obs_names.astype(str)
    candidate.var_names = candidate.var_names.astype(str)
    candidate.var_names_make_unique()

    if candidate.n_obs != len(selected_obs):
        raise RuntimeError(
            f"{cancer_type}: materialized {candidate.n_obs} cells but "
            f"selection metadata has {len(selected_obs)} rows."
        )
    if not candidate.obs_names.equals(selected_obs.index):
        selected_obs = selected_obs.reindex(candidate.obs_names)
        if selected_obs.isna().all(axis=1).any():
            raise RuntimeError(
                f"{cancer_type}: candidate obs order could not be aligned."
            )

    # Preserve raw serialized values and overwrite working columns with audited
    # canonical values used for this clustering run.
    candidate.obs[SOURCE_CANCER_COLUMN] = (
        selected_obs[SOURCE_CANCER_COLUMN].to_numpy()
    )
    candidate.obs[SOURCE_LABEL_COLUMN] = (
        selected_obs[SOURCE_LABEL_COLUMN].to_numpy()
    )
    candidate.obs["sample"] = (
        selected_obs["sample"].to_numpy()
    )
    candidate.obs["patient"] = (
        selected_obs["patient"].to_numpy()
    )
    candidate.obs["biopsy_stage"] = (
        selected_obs["biopsy_stage"].to_numpy()
    )
    candidate.obs["cancer_type"] = cancer_type
    candidate.obs[ANNOTATION_COLUMN] = (
        selected_obs[SELECTION_LABEL_COLUMN].to_numpy()
    )
    candidate.obs[QC_PASS_COLUMN] = (
        selected_obs[SELECTION_QC_COLUMN].to_numpy(dtype=bool)
    )

    if "counts" not in candidate.layers:
        raise KeyError(
            f"{cancer_type}: counts layer disappeared after subsetting."
        )

    raw_counts = candidate.layers["counts"]
    if not sp.issparse(raw_counts):
        raw_counts = sp.csr_matrix(raw_counts)
    else:
        raw_counts = sp.csr_matrix(raw_counts)

    raw_values = raw_counts.data
    if raw_values.size:
        if raw_values.min() < 0:
            raise ValueError(
                "Raw-count layer contains negative values."
            )
        sampled = raw_values[
            : min(len(raw_values), 1_000_000)
        ]
        fractional = float(
            np.max(
                np.abs(sampled - np.rint(sampled))
            )
        )
        if fractional > 1e-6:
            raise ValueError(
                f"Raw-count layer is not integer-like: {fractional}"
            )
        raw_counts.data = np.rint(
            raw_counts.data
        ).astype(np.uint32, copy=False)

    raw_counts.eliminate_zeros()
    raw_counts.sort_indices()
    candidate.layers["counts"] = raw_counts

    corrected = candidate.X
    if hasattr(corrected, "compute"):
        corrected = corrected.compute()
    corrected = np.asarray(
        corrected,
        dtype=np.float32,
    )

    if not np.isfinite(corrected).all():
        raise ValueError(
            "Corrected expression contains NaN/inf."
        )
    if corrected.min(initial=0) < 0:
        raise ValueError(
            "Corrected expression contains negative values."
        )

    candidate.X = np.log1p(
        corrected,
    ).astype(np.float32, copy=False)

    for column in [
        "sample",
        "patient",
        "cancer_type",
        "biopsy_stage",
        ANNOTATION_COLUMN,
        SOURCE_CANCER_COLUMN,
        SOURCE_LABEL_COLUMN,
    ]:
        candidate.obs[column] = categorical_string(
            candidate.obs[column]
        )

    raw_n_cells = np.asarray(
        (raw_counts > 0).sum(axis=0)
    ).reshape(-1)
    raw_total = np.asarray(
        raw_counts.sum(axis=0)
    ).reshape(-1)

    candidate.var["raw_n_cells_by_counts"] = raw_n_cells
    candidate.var["raw_total_counts"] = raw_total

    minimum_raw_cells = max(
        int(MIN_RAW_CELLS_PER_GENE),
        int(
            math.ceil(
                candidate.n_obs
                * MIN_RAW_CELL_FRACTION_PER_GENE
            )
        ),
    )
    candidate.var["raw_supported_for_identity"] = (
        raw_n_cells >= minimum_raw_cells
    )

    genes = pd.Index(candidate.var_names.astype(str))
    upper = genes.str.upper()

    candidate.var["mt"] = upper.str.startswith(("MT-", "MT_"))
    candidate.var["ribo"] = upper.str.startswith(("RPS", "RPL"))
    candidate.var["hb"] = upper.isin(HEMOGLOBIN_GENES)
    candidate.var["cell_cycle_gene"] = upper.isin(
        S_PHASE_GENES | G2M_PHASE_GENES
    )

    deprecated = np.zeros(candidate.n_vars, dtype=bool)
    for column in [
        "deprecated",
        "is_deprecated",
        "feature_is_deprecated",
    ]:
        if column in candidate.var:
            values = candidate.var[column]
            if pd.api.types.is_bool_dtype(values):
                deprecated |= values.fillna(False).to_numpy()
            else:
                deprecated |= (
                    values.astype(str)
                    .str.lower()
                    .isin({"true", "1", "yes"})
                    .to_numpy()
                )
    candidate.var["deprecated_for_identity"] = deprecated

    technical = (
        candidate.var["mt"].to_numpy(dtype=bool)
        | candidate.var["ribo"].to_numpy(dtype=bool)
        | candidate.var["hb"].to_numpy(dtype=bool)
        | candidate.var["deprecated_for_identity"].to_numpy(dtype=bool)
    )
    if EXCLUDE_CELL_CYCLE_FROM_IDENTITY_PCA:
        technical |= candidate.var[
            "cell_cycle_gene"
        ].to_numpy(dtype=bool)

    candidate.var["technical_excluded_from_identity"] = technical

    gate_row = selection_gate_audit.loc[
        selection_gate_audit["cancer_type"].eq(cancer_type)
    ].iloc[0].to_dict()

    candidate.uns["tumor_unresolved_clustering"] = {
        "pipeline_version": PIPELINE_VERSION,
        "source_merged_zarr": str(MERGED_ZARR),
        "candidate_annotation_column": ANNOTATION_COLUMN,
        "candidate_labels": sorted(CANDIDATE_LABELS),
        "require_qc_pass": bool(REQUIRE_QC_PASS),
        "selection_cancer_source": "SAMPLE_INFO mapping",
        "selection_gate_audit": to_builtin(gate_row),
        "source_cancer_column": SOURCE_CANCER_COLUMN,
        "source_label_column": SOURCE_LABEL_COLUMN,
        "matrix_X": (
            "log1p ResolVI-corrected expression normalized to 10,000"
        ),
        "counts_layer": "observed raw integer Proseg counts",
        "minimum_raw_cells_per_identity_gene": int(minimum_raw_cells),
        "keratinocyte_signature_used_for_clustering": False,
    }

    print(candidate)
    print(
        "Cells by sample:",
        candidate.obs["sample"].value_counts().to_dict(),
    )
    print(
        "Cells by source label:",
        candidate.obs[ANNOTATION_COLUMN].value_counts().to_dict(),
    )
    return candidate


print("Candidate-loading helpers ready.")


Candidate-loading helpers ready.


In [7]:
# ---------------------------------------------------------------------
# Sample-balanced HVGs and identity PCA
# ---------------------------------------------------------------------
def sample_balanced_indices(
    obs: pd.DataFrame,
    *,
    max_total: int,
    max_per_sample: int,
    random_state: int,
) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    selected = []

    sample_values = obs["sample"].astype(str).to_numpy()
    for sample in sorted(pd.unique(sample_values)):
        indices = np.flatnonzero(
            sample_values == sample
        )
        if len(indices) > max_per_sample:
            indices = rng.choice(
                indices,
                size=max_per_sample,
                replace=False,
            )
        selected.append(indices)

    selected = np.concatenate(selected)
    if len(selected) > max_total:
        selected = rng.choice(
            selected,
            size=max_total,
            replace=False,
        )
    return np.sort(selected)


def choose_identity_features(
    candidate: ad.AnnData,
) -> dict:
    hvg_indices = sample_balanced_indices(
        candidate.obs,
        max_total=int(HVG_MAX_CELLS),
        max_per_sample=int(HVG_MAX_CELLS_PER_SAMPLE),
        random_state=int(RANDOM_STATE),
    )

    hvg_data = candidate[
        hvg_indices,
        :,
    ].copy()

    batch_key = (
        "sample"
        if hvg_data.obs["sample"].nunique() > 1
        else None
    )

    sc.pp.highly_variable_genes(
        hvg_data,
        flavor="seurat",
        n_top_genes=min(
            int(N_TOP_HVG),
            hvg_data.n_vars,
        ),
        batch_key=batch_key,
        subset=False,
        inplace=True,
    )

    highly_variable = (
        hvg_data.var["highly_variable"]
        .reindex(candidate.var_names)
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    raw_supported = candidate.var[
        "raw_supported_for_identity"
    ].to_numpy(dtype=bool)
    technical = candidate.var[
        "technical_excluded_from_identity"
    ].to_numpy(dtype=bool)

    integration_feature = (
        highly_variable
        & raw_supported
        & ~technical
    )

    if integration_feature.sum() < 500:
        warnings.warn(
            "Fewer than 500 features survived raw-support and technical "
            "filters. Relaxing raw-support requirement but retaining "
            "technical exclusions."
        )
        integration_feature = (
            highly_variable
            & ~technical
        )

    if integration_feature.sum() < 100:
        raise RuntimeError(
            "Insufficient identity features after filtering."
        )

    candidate.var["highly_variable_sample_aware"] = (
        highly_variable
    )
    candidate.var["integration_feature"] = (
        integration_feature
    )

    del hvg_data
    gc.collect()

    return {
        "n_hvg_cells_used": int(len(hvg_indices)),
        "n_highly_variable": int(highly_variable.sum()),
        "n_raw_supported": int(raw_supported.sum()),
        "n_technical_excluded": int(technical.sum()),
        "n_integration_features": int(
            integration_feature.sum()
        ),
    }


def run_identity_pca(
    candidate: ad.AnnData,
) -> dict:
    feature_mask = candidate.var[
        "integration_feature"
    ].to_numpy(dtype=bool)

    temp = ad.AnnData(
        X=np.ascontiguousarray(
            candidate.X[:, feature_mask],
            dtype=np.float32,
        ),
        obs=candidate.obs.copy(),
        var=candidate.var.loc[
            feature_mask
        ].copy(),
    )

    n_comps = min(
        int(PCA_N_COMPS),
        temp.n_obs - 1,
        temp.n_vars - 1,
    )
    if n_comps < 2:
        raise RuntimeError(
            f"Insufficient dimensions for PCA: {temp.shape}"
        )

    if RAPIDS_AVAILABLE:
        rsc.get.anndata_to_GPU(
            temp,
            convert_all=False,
        )
        call_with_supported_kwargs(
            rsc.pp.scale,
            temp,
            zero_center=True,
            max_value=float(PCA_SCALE_MAX_VALUE),
        )
        call_with_supported_kwargs(
            rsc.pp.pca,
            temp,
            n_comps=int(n_comps),
            zero_center=True,
            random_state=int(RANDOM_STATE),
            dtype="float32",
        )
        candidate.obsm["X_pca"] = numpy_from_gpu(
            temp.obsm["X_pca"]
        ).astype(np.float32, copy=False)
        candidate.uns["pca"] = to_builtin(
            temp.uns.get("pca", {})
        )
    else:
        if REQUIRE_RAPIDS and not ALLOW_CPU_FALLBACK:
            raise RuntimeError(
                "RAPIDS is required but unavailable."
            )
        sc.pp.scale(
            temp,
            zero_center=True,
            max_value=float(PCA_SCALE_MAX_VALUE),
        )
        sc.tl.pca(
            temp,
            n_comps=int(n_comps),
            zero_center=True,
            svd_solver="randomized",
            random_state=int(RANDOM_STATE),
        )
        candidate.obsm["X_pca"] = np.asarray(
            temp.obsm["X_pca"],
            dtype=np.float32,
        )
        candidate.uns["pca"] = to_builtin(
            temp.uns.get("pca", {})
        )

    del temp
    free_gpu_memory()

    return {
        "n_components": int(n_comps),
        "pca_shape": list(
            map(
                int,
                candidate.obsm["X_pca"].shape,
            )
        ),
    }


print("Feature-selection/PCA helpers ready.")


Feature-selection/PCA helpers ready.


In [8]:
# ---------------------------------------------------------------------
# Harmony, graph, UMAP, and requested Leiden resolutions
# ---------------------------------------------------------------------
def make_embedding_shell(
    candidate: ad.AnnData,
    representation_key: str,
) -> ad.AnnData:
    shell = ad.AnnData(
        X=np.zeros(
            (candidate.n_obs, 1),
            dtype=np.float32,
        ),
        obs=candidate.obs.copy(),
    )
    shell.obsm[representation_key] = np.asarray(
        candidate.obsm[representation_key],
        dtype=np.float32,
    )
    return shell


def run_harmony(
    candidate: ad.AnnData,
) -> dict:
    if not APPLY_HARMONY:
        candidate.obsm["X_pca_integrated"] = (
            candidate.obsm["X_pca"].copy()
        )
        return {
            "applied": False,
            "reason": "APPLY_HARMONY=False",
            "basis": "X_pca",
            "adjusted_basis": "X_pca_integrated",
        }

    if HARMONY_BATCH_KEY not in candidate.obs:
        raise KeyError(
            f"Harmony key {HARMONY_BATCH_KEY!r} is absent."
        )

    counts = (
        candidate.obs[HARMONY_BATCH_KEY]
        .astype(str)
        .value_counts()
    )
    if len(counts) < 2:
        candidate.obsm["X_pca_integrated"] = (
            candidate.obsm["X_pca"].copy()
        )
        return {
            "applied": False,
            "reason": (
                f"Only one {HARMONY_BATCH_KEY} category."
            ),
            "basis": "X_pca",
            "adjusted_basis": "X_pca_integrated",
        }

    shell = make_embedding_shell(
        candidate,
        "X_pca",
    )

    try:
        if RAPIDS_AVAILABLE:
            rsc.get.anndata_to_GPU(
                shell,
                convert_all=True,
            )
            call_with_supported_kwargs(
                rsc.pp.harmony_integrate,
                shell,
                key=HARMONY_BATCH_KEY,
                basis="X_pca",
                adjusted_basis="X_pca_integrated",
                dtype=np.float64,
                theta=float(HARMONY_THETA),
                tau=int(HARMONY_TAU),
                max_iter_harmony=int(
                    HARMONY_MAX_ITER
                ),
                random_state=int(RANDOM_STATE),
                verbose=False,
            )
            integrated = numpy_from_gpu(
                shell.obsm["X_pca_integrated"]
            ).astype(np.float32, copy=False)
        else:
            if REQUIRE_RAPIDS and not ALLOW_CPU_FALLBACK:
                raise RuntimeError(
                    "RAPIDS is required but unavailable."
                )
            import scanpy.external as sce

            sce.pp.harmony_integrate(
                shell,
                key=HARMONY_BATCH_KEY,
                basis="X_pca",
                adjusted_basis="X_pca_integrated",
                theta=float(HARMONY_THETA),
                max_iter_harmony=int(
                    HARMONY_MAX_ITER
                ),
                random_state=int(RANDOM_STATE),
            )
            integrated = np.asarray(
                shell.obsm["X_pca_integrated"],
                dtype=np.float32,
            )

        if integrated.shape != candidate.obsm["X_pca"].shape:
            raise RuntimeError(
                "Harmony embedding shape changed unexpectedly."
            )
        if not np.isfinite(integrated).all():
            raise RuntimeError(
                "Harmony embedding contains nonfinite values."
            )

        mean_abs_change = float(
            np.mean(
                np.abs(
                    integrated
                    - candidate.obsm["X_pca"]
                )
            )
        )
        candidate.obsm["X_pca_integrated"] = integrated

        return {
            "applied": True,
            "batch_key": HARMONY_BATCH_KEY,
            "category_counts": counts.to_dict(),
            "theta": float(HARMONY_THETA),
            "tau": int(HARMONY_TAU),
            "mean_absolute_change_from_pca": mean_abs_change,
            "basis": "X_pca",
            "adjusted_basis": "X_pca_integrated",
        }

    except Exception as exc:
        if FAIL_IF_HARMONY_FAILS:
            raise
        warnings.warn(
            "Harmony failed; clustering unintegrated PCA. "
            f"{type(exc).__name__}: {exc}"
        )
        candidate.obsm["X_pca_integrated"] = (
            candidate.obsm["X_pca"].copy()
        )
        return {
            "applied": False,
            "reason": f"{type(exc).__name__}: {exc}",
            "basis": "X_pca",
            "adjusted_basis": "X_pca_integrated",
        }

    finally:
        del shell
        free_gpu_memory()


def run_embedding_graph(
    candidate: ad.AnnData,
    *,
    representation_key: str,
    output_umap_key: str,
    run_leiden: bool,
) -> dict:
    shell = make_embedding_shell(
        candidate,
        representation_key,
    )

    n_pcs = min(
        int(N_PCS_NEIGHBORS),
        shell.obsm[representation_key].shape[1],
    )

    if RAPIDS_AVAILABLE:
        rsc.get.anndata_to_GPU(
            shell,
            convert_all=True,
        )
        call_with_supported_kwargs(
            rsc.pp.neighbors,
            shell,
            n_neighbors=int(N_NEIGHBORS),
            n_pcs=int(n_pcs),
            use_rep=representation_key,
            random_state=int(RANDOM_STATE),
            algorithm=NEIGHBOR_ALGORITHM,
            metric="euclidean",
        )
        call_with_supported_kwargs(
            rsc.tl.umap,
            shell,
            min_dist=float(UMAP_MIN_DIST),
            spread=float(UMAP_SPREAD),
            random_state=int(RANDOM_STATE),
        )

        leiden_keys = []
        if run_leiden:
            for resolution in LEIDEN_RESOLUTIONS:
                key = resolution_key(resolution)
                call_with_supported_kwargs(
                    rsc.tl.leiden,
                    shell,
                    resolution=float(resolution),
                    random_state=int(RANDOM_STATE),
                    key_added=key,
                    n_iterations=int(
                        LEIDEN_N_ITERATIONS
                    ),
                )
                leiden_keys.append(key)

        rsc.get.anndata_to_CPU(
            shell,
            convert_all=True,
        )
    else:
        if REQUIRE_RAPIDS and not ALLOW_CPU_FALLBACK:
            raise RuntimeError(
                "RAPIDS is required but unavailable."
            )
        sc.pp.neighbors(
            shell,
            n_neighbors=int(N_NEIGHBORS),
            n_pcs=int(n_pcs),
            use_rep=representation_key,
            random_state=int(RANDOM_STATE),
            metric="euclidean",
        )
        sc.tl.umap(
            shell,
            min_dist=float(UMAP_MIN_DIST),
            spread=float(UMAP_SPREAD),
            random_state=int(RANDOM_STATE),
        )

        leiden_keys = []
        if run_leiden:
            for resolution in LEIDEN_RESOLUTIONS:
                key = resolution_key(resolution)
                sc.tl.leiden(
                    shell,
                    resolution=float(resolution),
                    random_state=int(RANDOM_STATE),
                    key_added=key,
                    n_iterations=int(
                        LEIDEN_N_ITERATIONS
                    ),
                )
                leiden_keys.append(key)

    candidate.obsm[output_umap_key] = np.asarray(
        shell.obsm["X_umap"],
        dtype=np.float32,
    )

    if run_leiden:
        for key in leiden_keys:
            candidate.obs[key] = pd.Categorical(
                shell.obs[key].astype(str)
            )

        candidate.obsp["connectivities"] = (
            shell.obsp["connectivities"].copy()
        )
        candidate.obsp["distances"] = (
            shell.obsp["distances"].copy()
        )
        candidate.uns["neighbors"] = to_builtin(
            shell.uns.get("neighbors", {})
        )

    del shell
    free_gpu_memory()

    return {
        "representation": representation_key,
        "umap_key": output_umap_key,
        "n_neighbors": int(N_NEIGHBORS),
        "n_pcs": int(n_pcs),
        "leiden_keys": leiden_keys,
    }


print("Harmony/graph/Leiden helpers ready.")


Harmony/graph/Leiden helpers ready.


## Cluster diagnostics

The notebook does not choose a “best” resolution automatically. It provides:

```text
cluster count and size at every resolution
adjacent-resolution adjusted Rand index
sample composition
Screen/C2D15 composition
current Tumor/epithelial versus Other/unresolved composition
UMAPs for all ten resolutions
```

A useful resolution should show coherent expression programs without being
dominated by one sample, low-quality cells, cell cycle, or ResolVI diffusion.

The primary-resolution marker table and dotplot are aids for review; they do
not relabel cells.


In [9]:
# ---------------------------------------------------------------------
# Resolution summaries, marker audits, and figures
# ---------------------------------------------------------------------
AUDIT_MARKERS = {
    "melanoma": [
        "MLANA", "PMEL", "TYR", "DCT", "MITF", "SOX10",
        "S100B", "TYRP1", "PRAME", "MIA", "CSPG4",
        "AXL", "NGFR", "SOX9", "WNT5A", "FOSL1",
        # Naturally emerging keratinocyte/stratified epithelial cluster audit.
        "DSP", "DMKN", "DSG1", "DSC3",
        # Competing lineages.
        "PTPRC", "LST1", "COL1A1", "PECAM1",
    ],
    "NSCLC": [
        "EPCAM", "KRT8", "KRT18", "KRT19",
        "SLC34A2", "NAPSA", "SFTPA1", "SFTPA2",
        "SFTPB", "SFTPC", "CLDN18",
        "TP63", "KRT5", "KRT14", "DSG3", "DSC3",
        "SCGB1A1", "FOXJ1", "PIFO", "CAPS", "TPPP3",
        "CEACAM5", "CEACAM6", "MSLN", "GPRC5A",
        "PTPRC", "LST1", "COL1A1", "PECAM1",
    ],
    "colon_cancer": [
        "EPCAM", "KRT8", "KRT18", "KRT19",
        "CEACAM5", "CEACAM6", "MUC13", "TFF3",
        "PHGR1", "LGALS4", "SLC26A3", "FABP1",
        "CDX2", "SATB2", "PIGR", "REG4", "SPINK4",
        "FCGBP", "MUC2", "AGR2",
        "PTPRC", "LST1", "COL1A1", "PECAM1",
    ],
}


def summarize_resolutions(
    candidate: ad.AnnData,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    resolution_rows = []
    cluster_rows = []

    for resolution in LEIDEN_RESOLUTIONS:
        key = resolution_key(resolution)
        counts = (
            candidate.obs[key]
            .astype(str)
            .value_counts()
        )

        resolution_rows.append(
            {
                "resolution": float(resolution),
                "cluster_key": key,
                "n_clusters": int(len(counts)),
                "n_cells": int(candidate.n_obs),
                "smallest_cluster": int(counts.min()),
                "median_cluster_size": float(
                    counts.median()
                ),
                "largest_cluster": int(counts.max()),
                "n_clusters_lt_50_cells": int(
                    (counts < 50).sum()
                ),
                "fraction_cells_in_clusters_lt_50": float(
                    counts[counts < 50].sum()
                    / candidate.n_obs
                ),
            }
        )

        frame = candidate.obs[
            [
                key,
                "sample",
                "patient",
                "biopsy_stage",
                ANNOTATION_COLUMN,
            ]
        ].copy()
        frame[key] = frame[key].astype(str)

        for cluster, group in frame.groupby(
            key,
            observed=True,
        ):
            sample_counts = (
                group["sample"]
                .astype(str)
                .value_counts()
            )
            label_counts = (
                group[ANNOTATION_COLUMN]
                .astype(str)
                .value_counts()
            )

            cluster_rows.append(
                {
                    "resolution": float(resolution),
                    "cluster_key": key,
                    "cluster": str(cluster),
                    "n_cells": int(len(group)),
                    "fraction_of_candidates": float(
                        len(group) / candidate.n_obs
                    ),
                    "n_samples": int(
                        group["sample"].nunique()
                    ),
                    "n_patients": int(
                        group["patient"].nunique()
                    ),
                    "top_sample": str(
                        sample_counts.index[0]
                    ),
                    "top_sample_fraction": float(
                        sample_counts.iloc[0]
                        / len(group)
                    ),
                    "fraction_screen": float(
                        (
                            group["biopsy_stage"]
                            .astype(str)
                            == "Screen"
                        ).mean()
                    ),
                    "fraction_c2d15": float(
                        (
                            group["biopsy_stage"]
                            .astype(str)
                            == "C2D15"
                        ).mean()
                    ),
                    "fraction_tumor_epithelial": float(
                        label_counts.get(
                            "Tumor/epithelial",
                            0,
                        )
                        / len(group)
                    ),
                    "fraction_other_unresolved": float(
                        label_counts.get(
                            "Other/unresolved",
                            0,
                        )
                        / len(group)
                    ),
                }
            )

    return (
        pd.DataFrame(resolution_rows),
        pd.DataFrame(cluster_rows),
    )


def adjacent_resolution_ari(
    candidate: ad.AnnData,
) -> pd.DataFrame:
    rows = []
    for left, right in zip(
        LEIDEN_RESOLUTIONS[:-1],
        LEIDEN_RESOLUTIONS[1:],
    ):
        left_key = resolution_key(left)
        right_key = resolution_key(right)
        rows.append(
            {
                "resolution_1": float(left),
                "resolution_2": float(right),
                "cluster_key_1": left_key,
                "cluster_key_2": right_key,
                "adjusted_rand_index": float(
                    adjusted_rand_score(
                        candidate.obs[left_key].astype(str),
                        candidate.obs[right_key].astype(str),
                    )
                ),
            }
        )
    return pd.DataFrame(rows)


def composition_long(
    candidate: ad.AnnData,
    group_column: str,
) -> pd.DataFrame:
    rows = []
    for resolution in LEIDEN_RESOLUTIONS:
        key = resolution_key(resolution)
        table = (
            candidate.obs.groupby(
                [
                    key,
                    group_column,
                ],
                observed=True,
            )
            .size()
            .rename("n_cells")
            .reset_index()
        )
        table[key] = table[key].astype(str)
        table[group_column] = (
            table[group_column].astype(str)
        )
        totals = table.groupby(key)[
            "n_cells"
        ].transform("sum")
        table["fraction_within_cluster"] = (
            table["n_cells"] / totals
        )
        table.insert(
            0,
            "resolution",
            float(resolution),
        )
        table.insert(
            1,
            "cluster_key",
            key,
        )
        table = table.rename(
            columns={key: "cluster"}
        )
        rows.append(table)
    return pd.concat(
        rows,
        ignore_index=True,
    )


def downsample_plot_indices(
    n_cells: int,
    max_cells: int,
) -> np.ndarray:
    if n_cells <= max_cells:
        return np.arange(n_cells)
    rng = np.random.default_rng(RANDOM_STATE)
    return np.sort(
        rng.choice(
            n_cells,
            size=max_cells,
            replace=False,
        )
    )


def category_colors(
    values: pd.Series,
):
    categories = sorted(
        pd.unique(
            values.astype(str)
        )
    )
    cmap = plt.get_cmap(
        "turbo",
        max(len(categories), 2),
    )
    mapping = {
        category: cmap(index)
        for index, category in enumerate(categories)
    }
    return mapping


def save_resolution_umap_grid(
    candidate: ad.AnnData,
    path: Path,
    cancer_type: str,
):
    indices = downsample_plot_indices(
        candidate.n_obs,
        PLOT_MAX_CELLS,
    )
    coordinates = candidate.obsm["X_umap"][
        indices
    ]

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(25, 10),
    )
    for ax, resolution in zip(
        axes.flat,
        LEIDEN_RESOLUTIONS,
    ):
        key = resolution_key(resolution)
        values = candidate.obs[key].iloc[
            indices
        ].astype(str)
        mapping = category_colors(values)
        colors = [
            mapping[value]
            for value in values
        ]

        ax.scatter(
            coordinates[:, 0],
            coordinates[:, 1],
            c=colors,
            s=1.0,
            linewidths=0,
            alpha=0.75,
        )

        # Label cluster centroids.
        plot_frame = pd.DataFrame(
            {
                "x": coordinates[:, 0],
                "y": coordinates[:, 1],
                "cluster": values.to_numpy(),
            }
        )
        centroids = (
            plot_frame.groupby(
                "cluster",
                observed=True,
            )[["x", "y"]]
            .median()
        )
        for cluster, row in centroids.iterrows():
            ax.text(
                row["x"],
                row["y"],
                str(cluster),
                fontsize=7,
                ha="center",
                va="center",
                weight="bold",
            )

        ax.set_title(
            f"resolution {resolution:.1f} "
            f"({candidate.obs[key].nunique()} clusters)"
        )
        ax.axis("off")

    fig.suptitle(
        f"{cancer_type}: Tumor/epithelial + Other/unresolved Leiden grid",
        fontsize=16,
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_discrete_umap(
    candidate: ad.AnnData,
    *,
    basis_key: str,
    columns: list[str],
    path: Path,
    title: str,
):
    indices = downsample_plot_indices(
        candidate.n_obs,
        PLOT_MAX_CELLS,
    )
    coordinates = candidate.obsm[basis_key][
        indices
    ]

    fig, axes = plt.subplots(
        1,
        len(columns),
        figsize=(7 * len(columns), 6),
        squeeze=False,
    )

    for ax, column in zip(
        axes.flat,
        columns,
    ):
        values = candidate.obs[column].iloc[
            indices
        ].astype(str)
        mapping = category_colors(values)

        for category, color in mapping.items():
            mask = values.to_numpy() == category
            ax.scatter(
                coordinates[mask, 0],
                coordinates[mask, 1],
                s=1.2,
                linewidths=0,
                alpha=0.70,
                color=color,
                label=category,
            )
        ax.set_title(column)
        ax.axis("off")
        ax.legend(
            fontsize=7,
            frameon=False,
            markerscale=4,
            loc="best",
        )

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_primary_spatial_maps(
    candidate: ad.AnnData,
    cancer_type: str,
    output_dir: Path,
):
    key = resolution_key(
        PRIMARY_RESOLUTION
    )
    all_clusters = candidate.obs[key].astype(str)
    mapping = category_colors(all_clusters)

    for sample in sorted(
        candidate.obs["sample"].astype(str).unique()
    ):
        mask = (
            candidate.obs["sample"].astype(str)
            == sample
        ).to_numpy()
        spatial = np.asarray(
            candidate.obsm["spatial"][mask],
            dtype=float,
        )
        values = candidate.obs.loc[
            mask,
            key,
        ].astype(str)

        colors = [
            mapping[value]
            for value in values
        ]

        fig, ax = plt.subplots(
            figsize=(8, 8)
        )
        ax.scatter(
            spatial[:, 0],
            spatial[:, 1],
            c=colors,
            s=1.2,
            linewidths=0,
            alpha=0.75,
        )
        ax.set_aspect(
            "equal",
            adjustable="box",
        )
        ax.invert_yaxis()
        ax.set_title(
            f"{cancer_type} | {sample} | "
            f"Leiden {PRIMARY_RESOLUTION:.1f}"
        )
        ax.set_xlabel("spatial x")
        ax.set_ylabel("spatial y")
        fig.tight_layout()
        fig.savefig(
            output_dir
            / (
                f"{cancer_slug(cancer_type)}_"
                f"{sample}_spatial_leiden_"
                f"{PRIMARY_RESOLUTION:.1f}.png"
            ),
            dpi=PLOT_DPI,
            bbox_inches="tight",
        )
        plt.close(fig)


def balanced_marker_subset(
    candidate: ad.AnnData,
    cluster_key: str,
) -> ad.AnnData:
    rng = np.random.default_rng(RANDOM_STATE)
    labels = candidate.obs[
        cluster_key
    ].astype(str).to_numpy()
    selected = []

    for cluster in sorted(pd.unique(labels)):
        indices = np.flatnonzero(
            labels == cluster
        )
        if len(indices) > MAX_MARKER_CELLS_PER_CLUSTER:
            indices = rng.choice(
                indices,
                size=MAX_MARKER_CELLS_PER_CLUSTER,
                replace=False,
            )
        selected.append(indices)

    selected = np.sort(
        np.concatenate(selected)
    )
    return candidate[selected].copy()


def run_primary_marker_ranking(
    candidate: ad.AnnData,
    path: Path,
) -> pd.DataFrame:
    key = resolution_key(
        PRIMARY_RESOLUTION
    )
    marker_data = balanced_marker_subset(
        candidate,
        key,
    )

    sc.tl.rank_genes_groups(
        marker_data,
        groupby=key,
        method="wilcoxon",
        n_genes=min(
            int(N_MARKER_GENES),
            marker_data.n_vars,
        ),
        pts=True,
        use_raw=False,
        key_added="rank_genes_primary",
    )

    marker_table = sc.get.rank_genes_groups_df(
        marker_data,
        group=None,
        key="rank_genes_primary",
    )
    marker_table.insert(
        0,
        "resolution",
        float(PRIMARY_RESOLUTION),
    )
    marker_table.to_csv(
        path,
        index=False,
    )

    del marker_data
    gc.collect()
    return marker_table


def raw_marker_detection_table(
    candidate: ad.AnnData,
    cancer_type: str,
) -> pd.DataFrame:
    key = resolution_key(
        PRIMARY_RESOLUTION
    )
    requested = AUDIT_MARKERS[cancer_type]
    present = [
        gene
        for gene in requested
        if gene in candidate.var_names
    ]
    if not present:
        return pd.DataFrame()

    raw = candidate[
        :,
        present,
    ].layers["counts"]
    raw = (
        raw.tocsr()
        if sp.issparse(raw)
        else sp.csr_matrix(raw)
    )

    labels = candidate.obs[
        key
    ].astype(str).to_numpy()
    rows = []

    for cluster in sorted(pd.unique(labels)):
        mask = labels == cluster
        subset = raw[mask]
        fractions = np.asarray(
            (subset > 0).mean(axis=0)
        ).reshape(-1)
        means = np.asarray(
            subset.mean(axis=0)
        ).reshape(-1)

        for gene, fraction, mean in zip(
            present,
            fractions,
            means,
        ):
            rows.append(
                {
                    "resolution": float(
                        PRIMARY_RESOLUTION
                    ),
                    "cluster": str(cluster),
                    "gene": gene,
                    "raw_detection_fraction": float(
                        fraction
                    ),
                    "raw_mean_count": float(mean),
                    "n_cells": int(mask.sum()),
                }
            )

    return pd.DataFrame(rows)


def save_marker_dotplot(
    candidate: ad.AnnData,
    cancer_type: str,
    path: Path,
):
    key = resolution_key(
        PRIMARY_RESOLUTION
    )
    genes = [
        gene
        for gene in AUDIT_MARKERS[cancer_type]
        if gene in candidate.var_names
    ]
    if not genes:
        return

    dotplot = sc.pl.dotplot(
        candidate,
        var_names=genes,
        groupby=key,
        use_raw=False,
        standard_scale="var",
        dendrogram=False,
        show=False,
        return_fig=True,
    )
    dotplot.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close("all")


print("Diagnostic/plot helpers ready.")


Diagnostic/plot helpers ready.


In [10]:
# ---------------------------------------------------------------------
# Process one cancer type
# ---------------------------------------------------------------------
def process_cancer_type(
    cancer_type: str,
) -> dict:
    paths = cancer_paths(cancer_type)

    if (
        USE_EXISTING_COMPLETED
        and paths["success"].exists()
        and paths["assignments"].exists()
        and (
            paths["zarr"].exists()
            or paths["h5ad"].exists()
        )
        and not OVERWRITE_OUTPUTS
    ):
        print(
            "Reusing completed result:",
            cancer_type,
        )
        return json.loads(
            paths["success"].read_text()
        )

    started = time.time()
    paths["failure"].unlink(missing_ok=True)

    candidate = load_candidate_cancer_type(
        cancer_type
    )

    feature_info = choose_identity_features(
        candidate
    )
    candidate.var.loc[
        :,
        [
            "highly_variable_sample_aware",
            "raw_supported_for_identity",
            "technical_excluded_from_identity",
            "integration_feature",
            "raw_n_cells_by_counts",
            "raw_total_counts",
        ],
    ].to_csv(
        paths["features"]
    )

    pca_info = run_identity_pca(
        candidate
    )

    unintegrated_info = {
        "applied": False,
    }
    if SAVE_UNINTEGRATED_UMAP:
        unintegrated_info = run_embedding_graph(
            candidate,
            representation_key="X_pca",
            output_umap_key="X_umap_unintegrated",
            run_leiden=False,
        )

    harmony_info = run_harmony(
        candidate
    )

    graph_info = run_embedding_graph(
        candidate,
        representation_key="X_pca_integrated",
        output_umap_key="X_umap",
        run_leiden=True,
    )

    # Exact requested cluster columns must all exist.
    leiden_keys = [
        resolution_key(resolution)
        for resolution in LEIDEN_RESOLUTIONS
    ]
    missing_keys = [
        key
        for key in leiden_keys
        if key not in candidate.obs
    ]
    if missing_keys:
        raise RuntimeError(
            f"Missing Leiden columns: {missing_keys}"
        )

    resolution_summary, cluster_summary = (
        summarize_resolutions(candidate)
    )
    ari_table = adjacent_resolution_ari(
        candidate
    )
    sample_composition = composition_long(
        candidate,
        "sample",
    )
    label_composition = composition_long(
        candidate,
        ANNOTATION_COLUMN,
    )

    resolution_summary.to_csv(
        paths["resolution_summary"],
        index=False,
    )
    cluster_summary.to_csv(
        paths["cluster_summary"],
        index=False,
    )
    ari_table.to_csv(
        paths["ari"],
        index=False,
    )
    sample_composition.to_csv(
        paths["sample_composition"],
        index=False,
    )
    label_composition.to_csv(
        paths["label_composition"],
        index=False,
    )

    assignment_columns = [
        column
        for column in [
            "source_cell_id",
            "sample",
            "patient",
            "cancer_type",
            "biopsy_stage",
            ANNOTATION_COLUMN,
            QC_PASS_COLUMN,
            "tumor_epithelial_confidence_tier",
            "qc_total_counts",
            "qc_n_genes_by_counts",
            "qc_pct_counts_mt",
            "resolvi_diffusion_proportion",
            "S_score",
            "G2M_score",
            "phase",
            *leiden_keys,
        ]
        if column in candidate.obs
    ]
    assignments = candidate.obs[
        assignment_columns
    ].copy()
    assignments.index.name = "cell_id"
    assignments.to_parquet(
        paths["assignments"]
    )

    save_resolution_umap_grid(
        candidate,
        paths["umap_resolutions"],
        cancer_type,
    )

    audit_columns = [
        column
        for column in [
            "sample",
            "biopsy_stage",
            ANNOTATION_COLUMN,
            resolution_key(
                PRIMARY_RESOLUTION
            ),
        ]
        if column in candidate.obs
    ]
    save_discrete_umap(
        candidate,
        basis_key="X_umap",
        columns=audit_columns,
        path=paths["umap_audit"],
        title=(
            f"{cancer_type}: sample-Harmony tumor/unresolved UMAP"
            if harmony_info.get("applied", False)
            else f"{cancer_type}: unintegrated tumor/unresolved UMAP"
        ),
    )

    save_primary_spatial_maps(
        candidate,
        cancer_type,
        paths["spatial"],
    )

    marker_table_rows = 0
    if RUN_PRIMARY_MARKER_RANKING:
        marker_table = run_primary_marker_ranking(
            candidate,
            paths["marker_table"],
        )
        marker_table_rows = int(
            len(marker_table)
        )
        del marker_table

    raw_detection = raw_marker_detection_table(
        candidate,
        cancer_type,
    )
    raw_detection.to_csv(
        paths["raw_marker_detection"],
        index=False,
    )

    save_marker_dotplot(
        candidate,
        cancer_type,
        paths["dotplot"],
    )

    candidate.uns[
        "tumor_unresolved_clustering"
    ].update(
        {
            "feature_selection": feature_info,
            "pca": pca_info,
            "unintegrated_view": unintegrated_info,
            "harmony": harmony_info,
            "graph": graph_info,
            "leiden_resolutions": [
                float(value)
                for value in LEIDEN_RESOLUTIONS
            ],
            "primary_resolution": float(
                PRIMARY_RESOLUTION
            ),
            "primary_cluster_key": resolution_key(
                PRIMARY_RESOLUTION
            ),
            "marker_panel_used_for_clustering": False,
        }
    )

    if WRITE_FULL_ZARR:
        if paths["zarr"].exists():
            if OVERWRITE_OUTPUTS:
                shutil.rmtree(paths["zarr"])
            else:
                raise FileExistsError(
                    paths["zarr"]
                )
        candidate.write_zarr(
            paths["zarr"],
            chunks=(
                min(
                    int(ZARR_CHUNKS[0]),
                    candidate.n_obs,
                ),
                min(
                    int(ZARR_CHUNKS[1]),
                    candidate.n_vars,
                ),
            ),
        )

        # Lightweight read-back validation.
        check = ad.experimental.read_lazy(
            str(paths["zarr"])
        )
        if check.shape != candidate.shape:
            raise RuntimeError(
                f"Written Zarr shape changed: "
                f"{check.shape} vs {candidate.shape}"
            )

    if WRITE_H5AD:
        if (
            paths["h5ad"].exists()
            and not OVERWRITE_OUTPUTS
        ):
            raise FileExistsError(
                paths["h5ad"]
            )
        safe_write_h5ad(
            candidate,
            paths["h5ad"],
        )

    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "cancer_type": cancer_type,
        "n_cells": int(candidate.n_obs),
        "n_genes": int(candidate.n_vars),
        "n_samples": int(
            candidate.obs["sample"].nunique()
        ),
        "n_patients": int(
            candidate.obs["patient"].nunique()
        ),
        "candidate_label_counts": (
            candidate.obs[ANNOTATION_COLUMN]
            .astype(str)
            .value_counts()
            .to_dict()
        ),
        "feature_selection": feature_info,
        "pca": pca_info,
        "harmony": harmony_info,
        "graph": graph_info,
        "primary_resolution": float(
            PRIMARY_RESOLUTION
        ),
        "primary_cluster_key": resolution_key(
            PRIMARY_RESOLUTION
        ),
        "primary_n_clusters": int(
            candidate.obs[
                resolution_key(
                    PRIMARY_RESOLUTION
                )
            ].nunique()
        ),
        "marker_table_rows": int(
            marker_table_rows
        ),
        "output_zarr": (
            str(paths["zarr"])
            if WRITE_FULL_ZARR
            else ""
        ),
        "output_h5ad": (
            str(paths["h5ad"])
            if WRITE_H5AD
            else ""
        ),
        "assignments": str(
            paths["assignments"]
        ),
        "runtime_minutes": float(
            (time.time() - started) / 60
        ),
    }

    write_json(
        summary,
        paths["summary"],
    )
    write_json(
        summary,
        paths["success"],
    )

    print(
        f"Completed {cancer_type}: "
        f"{candidate.n_obs:,} cells, "
        f"{summary['primary_n_clusters']} clusters "
        f"at resolution {PRIMARY_RESOLUTION:.1f}"
    )

    del candidate
    free_gpu_memory()
    return summary


print("Cancer-type processor ready.")


Cancer-type processor ready.


In [11]:
# ---------------------------------------------------------------------
# Run selected cancer types
# ---------------------------------------------------------------------
results = {}
failures = {}

for cancer_type in CANCER_TYPES_TO_RUN:
    print("\n" + "=" * 90)
    print("Cancer type:", cancer_type)
    print("=" * 90)

    try:
        results[cancer_type] = process_cancer_type(
            cancer_type
        )
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        failures[cancer_type] = error
        paths = cancer_paths(cancer_type)
        write_json(
            {
                "pipeline_version": PIPELINE_VERSION,
                "cancer_type": cancer_type,
                "error": error,
                "traceback": traceback.format_exc(),
            },
            paths["failure"],
        )
        traceback.print_exc(limit=30)
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        plt.close("all")
        free_gpu_memory()

summary_table = pd.DataFrame(
    list(results.values())
)
run_tag = "__".join(
    cancer_slug(value)
    for value in CANCER_TYPES_TO_RUN
)
summary_table.to_csv(
    OUTPUT_ROOT / f"run_{run_tag}_summary.csv",
    index=False,
)
write_json(
    failures,
    OUTPUT_ROOT / f"run_{run_tag}_failures.json",
)

display(summary_table)
print("Completed:", sorted(results))
print("Failures:", json.dumps(failures, indent=2))

if failures:
    raise RuntimeError(
        "At least one cancer-type clustering run failed. "
        "Completed outputs were preserved."
    )



Cancer type: NSCLC
NSCLC: materializing 140,501 candidate cells (140,501 before QC)...
AnnData object with n_obs × n_vars = 140501 × 14887
    obs: 'source_cell_id', 'sample', 'patient', 'cancer_type', 'biopsy_stage', 'prelim_cell_type_primary_tumor_expanded', 'prelim_T_confidence_tier', 'prelim_T_confidence_rank', 'prelim_Treg_confidence_tier', 'prelim_Treg_confidence_rank', 'prelim_T_primary', 'prelim_CD4_T', 'prelim_CD8_T', 'prelim_Treg_supported', 'prelim_Treg_high_confidence', 'rescue_T_mixing_status', 'rescue_T_percentile_minus_max_nonT', 'tumor_epithelial_confidence_tier', 'mt_high', 'low_genes', 'qc_pass', 'S_score', 'G2M_score', 'phase', 'cell_cycle_scored', 'resolvi_diffusion_proportion', 'qc_total_counts', 'qc_n_genes_by_counts', 'qc_pct_counts_mt', 'qc_pct_counts_ribo', 'qc_pct_counts_hb', 'source_cancer_type_before_selection_normalization', 'source_annotation_before_selection_normalization'
    var: 'mt', 'ribo', 'hb', 'is_deprecated', 'gene_symbol', 'raw_n_cells_by_count

/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Completed NSCLC: 140,501 cells, 18 clusters at resolution 0.6


,pipeline_version,cancer_type,n_cells,n_genes,n_samples,n_patients,candidate_label_counts,feature_selection,pca,harmony,graph,primary_resolution,primary_cluster_key,primary_n_clusters,marker_table_rows,output_zarr,output_h5ad,assignments,runtime_minutes
0,2026-08-20-tumor-unresolved-cancer-specific-le...,NSCLC,140501,14887,4,2,"{'Other/unresolved': 115739, 'Tumor/epithelial...","{'n_hvg_cells_used': 64090, 'n_highly_variable...","{'n_components': 50, 'pca_shape': [140501, 50]}","{'applied': True, 'batch_key': 'sample', 'cate...","{'representation': 'X_pca_integrated', 'umap_k...",0.6,tumor_leiden_res_0_6,18,1800,/host_root/nethome/reny28/Projects/Visium_proj...,,/host_root/nethome/reny28/Projects/Visium_proj...,10.815317


Completed: ['NSCLC']
Failures: {}


# Reading and reviewing the results

## Load one completed cancer-type object

```python
import anndata as ad

melanoma = ad.experimental.read_lazy(
    OUTPUT_ROOT
    / "melanoma"
    / "melanoma_tumor_unresolved_multires_leiden.zarr"
)

melanoma.obs[
    [
        "sample",
        "prelim_cell_type_primary_tumor_expanded",
        "tumor_leiden_res_0_1",
        "tumor_leiden_res_0_2",
        "tumor_leiden_res_0_3",
        "tumor_leiden_res_0_4",
        "tumor_leiden_res_0_5",
        "tumor_leiden_res_0_6",
        "tumor_leiden_res_0_7",
        "tumor_leiden_res_0_8",
        "tumor_leiden_res_0_9",
        "tumor_leiden_res_1_0",
    ]
].to_memory()
```

## Join cluster assignments back to another object

```python
assignments = pd.read_parquet(
    OUTPUT_ROOT
    / "melanoma"
    / "tables"
    / "melanoma_tumor_unresolved_cluster_assignments.parquet"
)

full_adata.obs = full_adata.obs.join(
    assignments[
        [
            "tumor_leiden_res_0_6",
        ]
    ],
    how="left",
)
```

## Suggested review order

1. `candidate_inventory.csv`
2. all-resolution UMAP grid
3. resolution summary and adjacent-resolution ARI
4. sample and Screen/C2D15 composition
5. source-label composition
6. primary marker table and raw-detection audit
7. per-sample spatial maps
8. unintegrated versus Harmony UMAP audit

## Important caution

Clusters can represent:

```text
normal epithelial identities
malignant tumor states
treatment-associated states
cell cycle or stress
sample/patient effects
residual non-tumor contamination
low-information or highly diffused cells
```

Leiden itself does not establish malignancy. Keep the current broad label until
cluster marker programs, raw detection, spatial localization, sample balance,
and pathology context support a more specific interpretation.
